# HW2: Deep Q-Network

We train a **DQN agent** to push a box to a goal position using low-dimensional state observations (`high_level_state`: end-effector xy, object xy, goal xy → 6-dim vector).

All implementation lives in `homework2.py`. This notebook imports from it, runs training, and visualises results.

## 1. Setup

We import hyperparameters and classes directly from `homework2.py` so that running `python homework2.py` and running this notebook produce identical behaviour.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from homework2 import (
    ReplayBuffer, DQNNetwork, DQNAgent, train,
    DEVICE, N_ACTIONS, STATE_DIM,
    MEMORY_SIZE, NUM_EPISODES, BATCH_SIZE,
    EPS_DECAY, EPS_END, EPS_START,
    GAMMA, LEARNING_RATE, TAU,
)

print(f"Using device: {DEVICE}")
print(f"episodes={NUM_EPISODES}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, tau={TAU}, gamma={GAMMA}")


## 2. Architecture & Design Choices

### Network
A 3-layer MLP: **6 → 64 → 64 → 8** (one Q-value per discrete action).  
ReLU activations — simple and effective for low-dimensional state spaces; no need for BatchNorm or dropout here.

### Double network (policy + target)
We maintain a **target network** whose weights are updated via soft update (`τ=0.005`) rather than hard copy. This decouples the TD target from the policy being updated, stabilising training.

$$\theta_{\text{target}} \leftarrow \tau\,\theta_{\text{policy}} + (1-\tau)\,\theta_{\text{target}}$$

### Optimizer: Adam (`lr=1e-4`)
Adam is chosen over SGD because it adapts per-parameter learning rates, which is important when gradients are noisy (sparse rewards, short episodes). `lr=1e-4` is conservative — prevents overshooting in early training when the replay buffer is small.

### Replay buffer (`capacity=10 000`)
Breaks temporal correlation between consecutive transitions. Capacity of 10k balances memory and diversity — large enough to sample varied experiences, small enough to stay recent.

### Epsilon-greedy exploration
Linear decay from `ε=0.9` to `ε=0.05` over **10 000 steps** (~200 episodes at max 50 steps each).  
High initial ε encourages broad exploration of the workspace before committing to a greedy policy.

### Reward shaping
The environment reward is:
$$r = \frac{1}{\max(100 \cdot d_{ee \to obj},\,1)} + \frac{1}{\max(100 \cdot d_{obj \to goal},\,1)}$$
This is a dense, shaped reward — both terms are positive at every step, which mitigates sparse-reward issues.

### Simulation speed (`n_splits=15`)
Each `env.step()` performs IK along a Cartesian trajectory split into `n_splits` sub-steps.  
We use 15 instead of the default 30 — halving simulation time with negligible impact on training quality (the agent only needs approximate contact, not smooth trajectories).

## 3. Smoke Tests

In [ ]:
buf = ReplayBuffer(100)
buf.push(np.zeros(6), 0, 1.0, np.ones(6), False)
assert len(buf) == 1
s, a, r, ns, d = buf.sample(1)
assert s.shape == (1, 6)
print("ReplayBuffer OK")

net = DQNNetwork(STATE_DIM, N_ACTIONS).to(DEVICE)
dummy = torch.zeros(1, STATE_DIM).to(DEVICE)
assert net(dummy).shape == (1, N_ACTIONS)
print("DQNNetwork OK")

agent_test = DQNAgent()
action = agent_test.select_action(np.zeros(6))
assert 0 <= action < N_ACTIONS
print("DQNAgent OK")


## 4. Training

The `train()` function:
- Automatically **resumes** from the latest checkpoint in `hw2_checkpoints/` if one exists for the given `run_name`
- Saves a checkpoint every `checkpoint_every` episodes
- Appends per-episode metrics (reward, RPS, loss, ε) to `hw2_{run_name}_log.csv`

Change `RUN_NAME` to `"run2"`, `"run3"` etc. for subsequent experiments.

In [ ]:
RUN_NAME = "run2"

agent, episode_rewards, episode_rps = train(
    run_name=RUN_NAME,
    n_splits=15,
    checkpoint_every=500,
)


## 5. Results

We plot total episode reward and reward-per-step (RPS) with a 50-episode smoothing window.  
RPS normalises for episode length and better reflects policy quality when episode durations vary.

In [ ]:
def smooth(data, window=50):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.3, color='steelblue', label='Raw')
axes[0].plot(range(49, len(episode_rewards)), smooth(episode_rewards), color='steelblue', label='Smoothed (50)')
axes[0].set_title('Episode Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()

axes[1].plot(episode_rps, alpha=0.3, color='darkorange', label='Raw')
axes[1].plot(range(49, len(episode_rps)), smooth(episode_rps), color='darkorange', label='Smoothed (50)')
axes[1].set_title('Reward Per Step (RPS)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('RPS')
axes[1].legend()

plt.tight_layout()
out_path = f'hw2_{RUN_NAME}_results.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f"Saved {out_path}")
